RAG with vector search using sqlitesearch.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # Load environment variables from .env file
openai_client = OpenAI()  # Initialize the OpenAI client

In [2]:
# Importing our sentence transformer model
from sentence_transformers import SentenceTransformer

# Creating a model object
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# importing our vector search index
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode='ivf',
    db_path="faq_vectors2.db"
)

In [4]:
query = 'How do i run kafka?'
query_vector = model.encode(query) # Always encode the query before vector searching

In [5]:
results = vs_index.search(query_vector, filter_dict={"course": "llm-zoomcamp"}, num_results=5)

In [6]:
results

[{'id': 'c2903069a0',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?',
  'answer': 'When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.\n\nIf you want to see what your Display name is, click on the "Edit Course Profile" button.\n\n<{IMAGE:image_1}>\n\n- **First field:** This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.\n- **Second field:** Change this to your official name as in your identification documents—passport, national ID card, driver\'s license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certific

In [ ]:
from rag_helper import RAGBase
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs): # stores an embedder instance (plus any RAGBase kwargs)
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [8]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client
)

In [9]:
vector_assistant.rag("can i still join?")

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'